In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_csv('GUG_data.csv')

# Identify columns with missing data
columns_with_missing = df.columns[df.isna().sum() > 0]

# Specify four columns without missing data
columns_without_missing = ['Year', 'Guardian Score/100', 'Institution', 'Subject']

# Columns to be predicted
columns_to_predict = [col for col in columns_with_missing if col not in columns_without_missing]

numeric_features = ['Year', 'Guardian Score/100']
categorical_features = ["Institution", "Subject"]

# Convert text data to numeric
label_encoders = {}
df_test = df.copy()
for col in categorical_features:
    le = LabelEncoder()
    df_test[col] = le.fit_transform(df_test[col].astype(str))
    label_encoders[col] = le

# Save results
results = {}

# Loop to process each column that has missing data
for target_col in columns_to_predict:
    print(f"Filling missing values for: {target_col}")

    # Training data (no missing values)
    train_df = df_test.dropna(subset=[target_col])

    # Test data (with missing values)
    test_df = df_test[df_test[target_col].isna()]

    if train_df.empty or test_df.empty:
        print(f"Skipping {target_col} due to insufficient data.")
        continue


    X_train, y_train = train_df[numeric_features + categorical_features], train_df[target_col]
    X_test = test_df[numeric_features + categorical_features]

    # Random Forest model to fill in missing data
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Predict missing values
    predicted_values = model.predict(X_test)

    # Replace missing values ​​with predictions
    df_test.loc[df_test[target_col].isna(), target_col] = predicted_values

    # --- Checking the accuracy of the model ---
    np.random.seed(42)

    # Select 20% of the data for testing
    non_missing_indices = train_df.index
    if len(non_missing_indices) < 5:
        print(f"Skipping MAE calculation for {target_col} due to insufficient non-missing values.")
        continue

    test_indices = np.random.choice(non_missing_indices, size=int(0.2 * len(non_missing_indices)), replace=False)

    true_values = df_test.loc[test_indices, target_col].copy()

    df_test.loc[test_indices, target_col] = np.nan

    predicted_values_test = model.predict(df_test.loc[test_indices, numeric_features + categorical_features])

    true_values_clean = true_values.dropna()
    predicted_values_clean = predicted_values_test[:len(true_values_clean)]

    print(f"Size of true_values_clean: {len(true_values_clean)}")
    print(f"Size of predicted_values_clean: {len(predicted_values_clean)}")

    if len(true_values_clean) > 0 and len(predicted_values_clean) > 0:
        mae_rf = mean_absolute_error(true_values_clean, predicted_values_clean)

        results[target_col] = {"Random Forest Imputation MAE": mae_rf}
    else:
        results[target_col] = {"Random Forest Imputation MAE": None}
        print(f"Not enough data to calculate MAE for {target_col}")

    df_test.loc[test_indices, target_col] = true_values

# Display results for each column
for col, res in results.items():
    print(f"Column: {col}")
    if res['Random Forest Imputation MAE'] is not None:
        print(f"Random Forest Imputation MAE: {res['Random Forest Imputation MAE']}")
    else:
        print("Not enough data to compute MAE.")
    print("-" * 50)

# Revert numeric values ​​to text for sorted columns
for col in categorical_features:
    df_test[col] = label_encoders[col].inverse_transform(df_test[col].astype(int))

# Round all numeric columns to one decimal place
numeric_columns = df_test.select_dtypes(include=[np.number]).columns
df_test[numeric_columns] = df_test[numeric_columns].round(1)

# Save the filled dataframe to the final CSV file
df_test.to_csv("final_filled_data.csv", index=False)

# Display filled data
print("Data after filling missing values:")
print(df_test)

Filling missing values for: % Satisfied with Teaching
Size of true_values_clean: 7687
Size of predicted_values_clean: 7687
Filling missing values for: % Satisfied with course
Size of true_values_clean: 7206
Size of predicted_values_clean: 7206
Filling missing values for: Continuation
Size of true_values_clean: 2438
Size of predicted_values_clean: 2438
Filling missing values for: Expenditure per student (fte)
Size of true_values_clean: 7000
Size of predicted_values_clean: 7000
Filling missing values for: Student: staff ratio
Size of true_values_clean: 7676
Size of predicted_values_clean: 7676
Filling missing values for: Career prospects
Size of true_values_clean: 6002
Size of predicted_values_clean: 6002
Filling missing values for: Value added score/10
Size of true_values_clean: 7410
Size of predicted_values_clean: 7410
Filling missing values for: Average Entry Tariff
Size of true_values_clean: 7640
Size of predicted_values_clean: 7640
Filling missing values for: % Satisfied with Assess